In [1]:
import numpy as np

import os

import xarray as xr

import glob

import matplotlib.pyplot as plt

import pandas as pd

from matplotlib.colors import BoundaryNorm

import matplotlib.colors as mcolors

import dask.array as da

import pickle

from scipy.stats import t

import matplotlib.ticker as ticker

import cartopy.crs as ccrs

import cartopy.feature as cfeature

import seaborn as sns

In [158]:
# directory where your files are saved

base_dir = "/home/annierosen16/advection_profiles_single_gridpoint_seager_vectorized"

# find all .pkl files in the directory

pkl_files = glob.glob(os.path.join(base_dir, "*_advection_profile.pkl"))

# load them into a dictionary

profiles = {}

for f in pkl_files:
    
    # extract the key, e.g., "12lst" from "12lst_advection_profile.pkl"
    
    key = os.path.basename(f).split("_")[0]
    
    with open(f, "rb") as handle:
        
        profiles[key] = pickle.load(handle)

In [33]:
# --- Function from earlier (with neighbor check) ---

def calculate_horizontal_advection_single_gridpoint(u, v, q, lat, lon, sfcp, pressure, time):
    
    R = 6371000.0   # Earth radius (m)
    
    seconds_to_day = 86400.0

    lat_rad = np.radians(lat)
    
    lon_rad = np.radians(lon)
    
    delta_phi = lat_rad[1] - lat_rad[0]
    
    delta_lambda = lon_rad[1] - lon_rad[0]

    lon2d, lat2d = np.meshgrid(lon_rad, lat_rad)
    
    cos_lat = np.cos(lat2d)

    nlev = u.shape[0]
    
    dqdx = np.zeros(nlev)
    
    dqdy = np.zeros(nlev)

    i = 1  # central lon index
    
    j = 1  # central lat index

    zonal_adv = np.full(nlev, np.nan)
    
    merid_adv = np.full(nlev, np.nan)

    for k in range(nlev):
       
        # --- check zonal neighbors ---
        
        lon_neighbors = [lon[i-1], lon[i], lon[i+1]]
        
        sp_vals_x = [
            
            float(sfcp.sel(valid_time=time, latitude=lat[j], longitude=ln, method="nearest").values)
            
            for ln in lon_neighbors
        ]
        
        zonal_ok = all(pressure[k] <= sp for sp in sp_vals_x)

        # --- check meridional neighbors ---
        
        lat_neighbors = [lat[j-1], lat[j], lat[j+1]]
        
        sp_vals_y = [
            
            float(sfcp.sel(valid_time=time, latitude=lt, longitude=lon[i], method="nearest").values)
            
            for lt in lat_neighbors
        ]
        
        merid_ok = all(pressure[k] <= sp for sp in sp_vals_y)
        
        if zonal_ok:
            
            dqdx[k] = (q[k, j, i+1] - q[k, j, i-1]) / (2 * delta_lambda)
            
            dqdx[k] /= (R * cos_lat[j, i])
            
            zonal_adv[k] = -(u[k, j, i] * dqdx[k]) * seconds_to_day

        if merid_ok:
            
            dqdy[k] = (q[k, j+1, i] - q[k, j-1, i]) / (2 * delta_phi)
            
            dqdy[k] /= R
            
            merid_adv[k] = -(v[k, j, i] * dqdy[k]) * seconds_to_day

    return zonal_adv, merid_adv

In [76]:
def calculate_horizontal_convergence_single_gridpoint(u, v, q, lat, lon, sfcp, pressure, time):
    
    """
    Calculate horizontal moisture convergence on a spherical grid.

    u, v, q: arrays of shape (levels, lat, lon)
    lat, lon: 1D arrays in degrees
    Returns: zonal and meridional convergence averaged over the domain per level
             Units: 1/day
    """
    
    R = 6371000.0   # Earth radius (m)
    
    seconds_to_day = 86400.0  # s/day

    # grid spacing (radians)
    
    lat_rad = np.radians(lat)
    
    lon_rad = np.radians(lon)
    
    delta_phi = lat_rad[1] - lat_rad[0]
    
    delta_lambda = lon_rad[1] - lon_rad[0]
    
    # meshgrid
    
    lon2d, lat2d = np.meshgrid(lon_rad, lat_rad)
    
    cos_lat = np.cos(lat2d)
    
    nlev = u.shape[0]
        
    v_cos = v * cos_lat[np.newaxis, :, :]

    # initialize derivatives
    
    nlev = u.shape[0]

    dudx = np.zeros(nlev)  # zonal derivative per level
    
    dvdy = np.zeros(nlev) 
    
    # lon center point
    
    i = 1 
    
    # lat center point
    
    j = 1
    
    zonal_conv = np.full(nlev, np.nan)
    
    merid_conv = np.full(nlev, np.nan)

    # compute derivatives for each pressure level
    
    for k in range(nlev):
        
                # --- check zonal neighbors ---
        
        lon_neighbors = [lon[i-1], lon[i], lon[i+1]]
        
        sp_vals_x = [
            
            float(sfcp.sel(valid_time=time, latitude=lat[j], longitude=ln, method="nearest").values)
            
            for ln in lon_neighbors
        ]
        
        zonal_ok = all(pressure[k] <= sp for sp in sp_vals_x)

        # --- check meridional neighbors ---
        
        lat_neighbors = [lat[j-1], lat[j], lat[j+1]]
        
        sp_vals_y = [
            
            float(sfcp.sel(valid_time=time, latitude=lt, longitude=lon[i], method="nearest").values)
            
            for lt in lat_neighbors
        ]
        
        merid_ok = all(pressure[k] <= sp for sp in sp_vals_y)
                
        if zonal_ok: 
        
            # zonal derivative of u (∂u/∂x)

            dudx[k] = (u[k, i, 2] - u[k, i, 0]) / (2 * delta_lambda)
            
            dudx[k] = dudx[k] / (R * cos_lat[i, j])
            
            zonal_conv[k] = -(q[k, i, j] * dudx[k]) * seconds_to_day 
            
        if merid_ok: 

            # meridional derivative of v (∂v/∂y)

            dvdy[k] = (v_cos[k, 2, j] - v_cos[k, 0, j]) / (2 * delta_phi)

            # convert to physical space (meters)

            dvdy[k] = dvdy[k] / (R * cos_lat[i, j])
            
            merid_conv[k] = -(q[k, i, j] * dvdy[k]) * seconds_to_day    
    
    return zonal_conv, merid_conv

In [82]:
# --- Synthetic grid ---

lat = np.array([30.0, 30.25, 30.5])   # 3 lats

lon = np.array([-100.0, -99.75, -99.5])  # 3 lons

plev = np.array([1000, 850, 700, 500, 300])  # hPa (pressure levels)

nlev = len(plev)

nlat = len(lat)

nlon = len(lon)

# --- Synthetic fields (random but smooth-ish) ---

rng = np.random.default_rng(0)

u = rng.normal(loc=5, scale=1, size=(nlev, nlat, nlon))    # m/s zonal wind

v = rng.normal(loc=0, scale=1, size=(nlev, nlat, nlon))    # m/s meridional wind

q = np.exp(-np.arange(nlev)[:, None, None] / 2.0) * rng.uniform(0, 0.02, size=(nlev, nlat, nlon))  # kg/kg, decreases with height

# --- Surface pressure (sfcp) ---

# make the central column "mountainous" (low sfcp) to force below-ground masking

sp_data = np.full((1, nlat, nlon), 950.0)  # hPa baseline

sp_data[0, :, 1] = 900.0  # low surface pressure at lon index 1 → below ground for high plev

sp_data[0, :, 0] = 900.0  # low surface pressure at lon index 1 → below ground for high plev

sfcp_sample = xr.DataArray(sp_data,
    
                    dims=("valid_time", "latitude", "longitude"),
    
                    coords={
        
                        "valid_time": [pd.Timestamp("2001-05-01")],
        
                        "latitude": lat,
        
                        "longitude": lon,
    
                    },
    
                    name="sp"

                   )

# --- Run test ---

time = pd.Timestamp("2001-05-01")

# zonal_adv, merid_adv = calculate_horizontal_advection_single_gridpoint(u, v, q, lat, lon, sfcp_sample, plev, time)

zonal_conv, merid_conv = calculate_horizontal_convergence_single_gridpoint(u, v, q, lat, lon, sfcp_sample, plev, time)

# print("Zonal advection:", zonal_adv)

# print("Meridional advection:", merid_adv)

print("")

print("Zonal convergence:", zonal_conv)

print("Meridional convergence:", merid_conv)

False
True
True
True
True

Zonal convergence: [            nan -1.26915906e-02  1.01824073e-02 -3.83840849e-03
 -8.07972082e-05]
Meridional convergence: [        nan  0.01587885  0.00240844 -0.01020293  0.00127434]


### surface term

In [153]:
def vertically_integrated_profile_in_pressure(uq, pressure, K, kk_plus1):
    
    pressure_limits = pressure[K : kk_plus1+1] 
    
    uq_limits = uq[K : kk_plus1+1]
            
    g = 9.81  # Gravitational acceleration in m/s^2
    
    rho_water = 1000  # Density of water in kg/m^3
    
    m_to_mm = 1000
                    
    dp = np.diff(pressure_limits)
                    
    integrated_value = (np.sum(0.5 * (uq_limits[:-1] + uq_limits[1:]) * dp) / (g * rho_water)) * m_to_mm

    return integrated_value

In [154]:
def find_limits_for_integration(sfcp, pressure):
    
    midpoint = 1
    
    eastpoint = 2
    
    westpoint = 0
    
    r = sfcp[midpoint, eastpoint]
    
    l = sfcp[midpoint, westpoint]
    
    Keast = np.argmax(pressure <= r)   # first index where pressure < east surface p
    
    Kwest = np.argmax(pressure <= l)  # first index where pressure < west surface p
    
    # First level above ground for both points
    
    kk = np.argmax(pressure <= min(r, l))
        
    # Next level (deeper, higher pressure)
    
    kk_plus1 = kk - 1   # if pressure is descending
    
    return(Keast, Kwest, kk_plus1)

In [155]:
pressure = np.array([1000, 925, 850, 700, 500, 300, 200, 100])

sfcp = np.array([
    [950, 970, 990],   # row 0
    [860, 960, 990],  # row 1 (our midpoint row)
    [900, 950, 970]    # row 2
])

nlev = len(pressure)

nlat, nlon = 3, 3

u_profile = np.linspace(2, 20, nlev)  # shape (nlev,)

u = np.zeros((nlev, nlat, nlon))

for k in range(nlev):
    
    u[k, :, :] = u_profile[k] + 0.5 * np.arange(nlon)  # small gradient eastward

q_surface = 0.014  # near surface

scale = 0.0005

q_profile = q_surface * np.exp(-np.arange(nlev) * scale * 200)  # decreasing with height

q = np.zeros((nlev, nlat, nlon))

for k in range(nlev):
    
    q[k, :, :] = q_profile[k] - 0.0001 * np.arange(nlat)[:, None]  # small gradient northward

Keast, Kwest, kk_plus1 = find_limits_for_integration(sfcp, pressure)

print(pressure[Keast], pressure[Kwest], pressure[kk_plus1])

925 850 925


In [156]:
# center latitude, west longitude (i+1) 

uq_west = u[:, 1, 0] * q[:, 1, 0]

uq_east = u[:, 1, 2] * q[:, 1, 2]

int_east = vertically_integrated_profile_in_pressure(uq_east, pressure, Keast, kk_plus1)

int_west = vertically_integrated_profile_in_pressure(uq_west, pressure, Kwest, kk_plus1)

print(int_east, int_west)

[]
[]
0.0 0.0
